# 📘 Notebook 15: Continuous Actions

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module D: Policy-Based Learning · Notebook 4 of 4 in this module · (15 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why value-based methods such as DQN cannot handle **continuous actions** directly
- Build a **Gaussian policy**: a network that outputs the mean of a normal distribution over actions, with a learned spread
- Adapt **PPO** to continuous actions and train it to swing up a pendulum
- Explain how the policy's **standard deviation** provides exploration, and watch it shrink during training
- Describe the idea behind **off-policy** methods for continuous control (DDPG, TD3, SAC)
- Compare an on-policy and an off-policy agent by the **amount of experience** each needs

> **Prerequisites:** Notebook 14 (PPO) and Notebook 09 (replay buffers and target networks, for the last section).
>
> 🔭 **Why this notebook matters later:** almost everything physical is continuous: the torque of a motor, the angle of a steering wheel, the dose of a drug. This notebook completes the set of methods of the course. Module E then turns to what changes when the agent plans ahead or faces an opponent.

> 📦 **Libraries used in this notebook.** **PyTorch**, **Gymnasium**, **imageio** and, for the last section, **Stable-Baselines3**. No GPU is needed. Training our own agent takes well under a minute. The library agent of Section 6 takes a few minutes.

In [ ]:
%pip install -q "gymnasium[classic-control]>=1.3" stable-baselines3 imageio imageio-ffmpeg torch

In [ ]:
import base64
import time
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
import torch
import torch.nn as nn
from IPython.display import HTML, display

torch.set_num_threads(2)


def show_video(frames, fps=30, width=300):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))


def network(n_inputs, n_outputs, hidden=64):
    return nn.Sequential(nn.Linear(n_inputs, hidden), nn.Tanh(),
                         nn.Linear(hidden, hidden), nn.Tanh(),
                         nn.Linear(hidden, n_outputs))

## 1. Actions that are numbers

### Intuition first
Every agent so far chose from a short list: left or right, one of four engines, flap or not. A driver does not choose between "steer left" and "steer right". They choose **how far** to turn the wheel, and any angle is possible.

### Formal definition
An action space is **discrete** when it is a finite list, and **continuous** when an action is one or more real numbers within given bounds. In Gymnasium the first is a `Discrete` space and the second a `Box`.

### Why it matters
DQN chooses its action with $\arg\max_a Q(s, a)$ and needs one network output per action. With a continuous action there are infinitely many candidates, and no list to take the maximum over.

One can cut the range into a few fixed values, as Notebook 08 cut the state into bins. That works for one coarse dimension and breaks down quickly: a robot arm with seven joints and only five torque levels per joint already has $5^7 = 78{,}125$ combined actions.

Policy-based methods do not have this problem. A policy network can output a number as easily as a probability.

## 2. The task: swinging up a pendulum

A rod hangs from a motor. The agent chooses the **torque** of the motor at every step, any value between −2 and 2, and should bring the rod upright and hold it there.

| | |
|---|---|
| **State** | 3 numbers: the cosine and the sine of the rod's angle, and its angular velocity |
| **Action** | 1 number: the torque, between −2 and 2 |
| **Reward** | negative at every step: the further from upright, the faster it moves and the more torque is used, the larger the penalty. The best possible reward of a step is 0. |
| **Episode** | always exactly 200 steps. There is no terminal state. |

The motor is too weak to lift the rod directly. The agent has to swing it back and forth to build up speed, and then catch it at the top.

In [ ]:
env = gym.make("Pendulum-v1")
print("observation space:", env.observation_space)
print("action space:     ", env.action_space)


def record(policy, seed=0):
    """One episode of the pendulum. `policy` maps a state to a torque. Returns frames and total reward."""
    video_env = gym.make("Pendulum-v1", render_mode="rgb_array")
    state, info = video_env.reset(seed=seed)
    frames, total = [video_env.render()], 0.0
    terminated = truncated = False
    while not (terminated or truncated):
        state, reward, terminated, truncated, info = video_env.step(policy(state))
        total += reward
        frames.append(video_env.render())
    return frames, total


frames, total = record(lambda state: env.action_space.sample())
print(f"random torques: total reward {total:.0f}")
show_video(frames[::2], fps=15)

A random agent collects a total reward of around −1,000 or worse. An agent that swings up quickly and holds the rod still reaches about −150, depending on where the rod starts.

The episode never **terminates**. It is always cut off after 200 steps, so every episode ends with `truncated`. That makes this environment a test of the rule we have followed since Notebook 05: after a truncation the last state still has a value, and the critic's estimate of it must be used.

## 3. A policy for continuous actions

### Intuition first
A darts player aims at a point and the dart lands somewhere near it. The aim is one number (or two), and the scatter around it is another. A beginner scatters widely, and so hits many different places: that is exploration. An expert scatters little.

### Formal definition
A **Gaussian policy** describes the action by a normal distribution:

$$\pi(a \mid s) = \mathcal{N}\big(\mu(s),\ \sigma^2\big)$$

- The network outputs the **mean** $\mu(s)$: where to aim in this state.
- The **standard deviation** $\sigma$ is a learned parameter: how much to scatter. We store its logarithm, so that $\sigma$ stays positive whatever value the optimiser gives it.

To act, the agent **samples** from the distribution. To learn, it needs the log-probability of the action it took, which the normal distribution provides in closed form. Nothing else in a policy-gradient method depends on the kind of action.

In [ ]:
class GaussianActor(nn.Module):
    """A policy for continuous actions: a network for the mean, one learned spread per action dimension."""

    def __init__(self, state_size, action_size, initial_std=1.0):
        super().__init__()
        self.mean = network(state_size, action_size)
        self.log_std = nn.Parameter(torch.full((action_size,), float(np.log(initial_std))))

    def distribution(self, states):
        return torch.distributions.Normal(self.mean(states), self.log_std.exp())


torch.manual_seed(0)
actor = GaussianActor(state_size=3, action_size=1)
state, info = env.reset(seed=0)

distribution = actor.distribution(torch.as_tensor(state))
samples = distribution.sample((5,)).squeeze(1)

print("mean:               ", distribution.mean.detach().numpy().round(3))
print("standard deviation: ", distribution.stddev.detach().numpy().round(3))
print("five sampled torques:", samples.numpy().round(2))
print("their log-probabilities:", distribution.log_prob(samples.unsqueeze(1)).squeeze(1).detach().numpy().round(2))

Two practical points.

- **Bounds.** A normal distribution can produce any number, but the motor accepts torques between −2 and 2. We simply clip the sampled action to the allowed range before passing it to the environment.
- **Several action dimensions.** A walking robot has one torque per joint. Each dimension then gets its own mean and its own spread, and the log-probability of the whole action is the **sum** over the dimensions. The code below is written for that general case.

## 4. PPO with a Gaussian policy

The algorithm is the PPO of Notebook 14. Two things change, and both are in the lines that touch the action:

- the distribution is `Normal` where it was `Categorical`,
- the log-probability is summed over the action dimensions.

The function also records the policy's standard deviation after every update, so that we can watch the exploration.

In [ ]:
def ppo_continuous(env_name, total_steps, n_envs=8, n_steps=256, epochs=10, batch_size=64, clip=0.2,
                   gamma=0.9, lam=0.95, learning_rate=1e-3, initial_std=1.0, seed=0, log=None, env_kwargs={}):
    """PPO for continuous actions. Returns the actor, the returns of finished episodes, and the spread over time."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    envs = [gym.make(env_name, **env_kwargs) for i in range(n_envs)]
    low, high = envs[0].action_space.low, envs[0].action_space.high
    states = np.array([env.reset(seed=seed * 1000 + i)[0] for i, env in enumerate(envs)], dtype=np.float32)
    state_size, action_size = states.shape[1], envs[0].action_space.shape[0]
    actor, critic = GaussianActor(state_size, action_size, initial_std), network(state_size, 1)
    parameters = list(actor.parameters()) + list(critic.parameters())
    optimiser = torch.optim.Adam(parameters, lr=learning_rate)
    finished_returns, running_returns, spreads, steps, updates = [], np.zeros(n_envs), [], 0, 0
    shape = (n_steps, n_envs)

    while steps < total_steps:
        # 1. collect a rollout
        S = np.zeros(shape + (state_size,), dtype=np.float32)
        A = np.zeros(shape + (action_size,), dtype=np.float32)
        R, T, D, V, NV = (np.zeros(shape, dtype=np.float32) for i in range(5))
        for t in range(n_steps):
            with torch.no_grad():
                batch = torch.as_tensor(states)
                actions = actor.distribution(batch).sample().numpy()
                V[t] = critic(batch).squeeze(1).numpy()
            S[t], A[t] = states, actions
            next_states = np.zeros_like(states)
            for i, env in enumerate(envs):
                state, reward, terminated, truncated, info = env.step(np.clip(actions[i], low, high))
                R[t, i], T[t, i], D[t, i] = reward, terminated, terminated or truncated
                next_states[i] = state
                running_returns[i] += reward
                if terminated or truncated:
                    finished_returns.append(running_returns[i])
                    running_returns[i] = 0.0
                    state, info = env.reset()
                states[i] = state
            with torch.no_grad():
                NV[t] = critic(torch.as_tensor(next_states)).squeeze(1).numpy()
        steps += n_steps * n_envs

        # 2. advantages with GAE (Notebook 13)
        advantages, running = np.zeros(shape, dtype=np.float32), np.zeros(n_envs)
        for t in reversed(range(n_steps)):
            delta = R[t] + gamma * (1 - T[t]) * NV[t] - V[t]
            running = delta + gamma * lam * (1 - D[t]) * running
            advantages[t] = running

        s = torch.as_tensor(S.reshape(-1, state_size))
        a = torch.as_tensor(A.reshape(-1, action_size))
        advantage = torch.as_tensor(advantages.reshape(-1))
        target = torch.as_tensor((advantages + V).reshape(-1))
        with torch.no_grad():
            old_log_probability = actor.distribution(s).log_prob(a).sum(dim=1)        # summed over action dimensions

        # 3. clipped updates (Notebook 14)
        for epoch in range(epochs):
            order = rng.permutation(len(s))
            for start in range(0, len(s), batch_size):
                index = torch.as_tensor(order[start:start + batch_size])
                log_probability = actor.distribution(s[index]).log_prob(a[index]).sum(dim=1)
                ratio = torch.exp(log_probability - old_log_probability[index])
                adv = advantage[index]
                adv = (adv - adv.mean()) / (adv.std() + 1e-8)
                actor_loss = -torch.min(ratio * adv, torch.clamp(ratio, 1 - clip, 1 + clip) * adv).mean()
                critic_loss = (critic(s[index]).squeeze(1) - target[index]).pow(2).mean()
                optimiser.zero_grad()
                (actor_loss + 0.5 * critic_loss).backward()
                nn.utils.clip_grad_norm_(parameters, 0.5)
                optimiser.step()

        spreads.append(actor.log_std.exp().mean().item())
        updates += 1
        if log and updates % log == 0 and finished_returns:
            print(f"step {steps:>7,}   average return of the last 20 episodes: {np.mean(finished_returns[-20:]):8.1f}   "
                  f"standard deviation of the policy: {spreads[-1]:.2f}")
    return actor, np.array(finished_returns), np.array(spreads)

The discount factor is 0.9, lower than usual. The pendulum's reward arrives at every step, so the agent does not need to look hundreds of steps ahead, and a shorter horizon gives the critic smaller numbers to fit. Exercise 3 tries the alternative.

In [ ]:
start = time.time()
actor, returns, spreads = ppo_continuous("Pendulum-v1", 80000, log=5)
print(f"\ntraining took {time.time() - start:.0f} seconds")

In [ ]:
def evaluate(policy, episodes=20):
    """Average total reward of a deterministic policy over 20 fixed starting positions."""
    totals = []
    for episode in range(episodes):
        state, info = env.reset(seed=70000 + episode)
        total, terminated, truncated = 0.0, False, False
        while not (terminated or truncated):
            state, reward, terminated, truncated, info = env.step(policy(state))
            total += reward
        totals.append(total)
    return float(np.mean(totals))


def mean_action(actor):
    """The deterministic version of a Gaussian policy: act at the mean, without scatter."""
    def policy(state):
        with torch.no_grad():
            return np.clip(actor.mean(torch.as_tensor(state)).numpy(), -2.0, 2.0)
    return policy


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(200 * np.arange(1, len(returns) + 1), returns, alpha=0.3)
axes[0].plot(200 * np.arange(20, len(returns) + 1), np.convolve(returns, np.ones(20) / 20, mode="valid"))
axes[0].set_title("PPO on the pendulum"); axes[0].set_xlabel("environment steps"); axes[0].set_ylabel("total reward of an episode")
axes[0].grid(True)
axes[1].plot(2048 * np.arange(1, len(spreads) + 1), spreads)
axes[1].set_title("Standard deviation of the policy"); axes[1].set_xlabel("environment steps"); axes[1].set_ylabel("sigma")
axes[1].grid(True)
plt.show()

print(f"trained policy, acting at its mean: average total reward {evaluate(mean_action(actor)):.0f}")
print(f"random policy:                      average total reward {evaluate(lambda state: env.action_space.sample()):.0f}")

The left plot shows the reward rising from the level of random play towards that of a good controller. The right plot shows something no earlier agent had: its **exploration schedule, learned by itself**. The policy starts with a standard deviation of 1, half the range of the motor, and narrows as it becomes sure of what to do. Nobody wrote a decay rule. The scatter shrinks because actions close to the mean turn out better than actions far from it.

In [ ]:
frames, total = record(mean_action(actor), seed=3)
print(f"trained agent: total reward {total:.0f}")
show_video(frames[::2], fps=15)

## 5. Looking at the policy

With one action dimension the learned policy can be drawn: the torque it applies, for each angle of the rod, when the rod is momentarily at rest.

In [ ]:
angles = np.linspace(-np.pi, np.pi, 181)
states = torch.as_tensor(np.stack([np.cos(angles), np.sin(angles), np.zeros_like(angles)], axis=1), dtype=torch.float32)
with torch.no_grad():
    torques = actor.mean(states).squeeze(1).clamp(-2, 2).numpy()

plt.figure(figsize=(8, 4))
plt.plot(np.degrees(angles), torques)
plt.axhline(0, color="gray", linestyle=":"); plt.axvline(0, color="gray", linestyle=":")
plt.title("The learned policy, for a rod at rest")
plt.xlabel("angle of the rod in degrees (0 = upright, ±180 = hanging down)"); plt.ylabel("torque applied")
plt.grid(True)
plt.show()

for degrees in (-20, -5, 0, 5, 20):
    index = int(np.argmin(np.abs(np.degrees(angles) - degrees)))
    print(f"rod at {degrees:+3d} degrees: torque {torques[index]:+.2f}")

Near the upright position (the middle of the plot) the torque changes sign as the angle does: the agent pushes against the lean, harder the further the rod has tipped. That is a proportional controller, the first thing an engineer would try, and the agent found it by trial and error. Further from the top the motor runs at its limit, where the agent is building up the swing.

A discrete agent could not produce this curve. It would have to jump between a few fixed torques.

## 6. Learning from every step: off-policy methods

PPO used 80,000 steps of experience for a problem with three state variables. For a simulated pendulum that costs seconds. For a real robot arm, where every step takes real time and wears real parts, it would be prohibitive.

Module C had the answer for discrete actions: a **replay buffer**, so that every step is learned from many times. Can continuous actions have that too? The obstacle was the `max` over actions. A family of algorithms gets round it with one idea.

### The idea
Train a critic that takes **both the state and the action** as inputs and outputs $Q(s, a)$. Such a critic is a differentiable function of the action. So we can ask it: "in which direction should this action change to increase its value?" and move the actor's output that way. The actor does not search for the best action. It is **pushed** towards it by the gradient of the critic.

| Algorithm | Year | What it adds |
|---|---|---|
| **DDPG** | 2015 | the idea above, with the replay buffer and target networks of DQN |
| **TD3** | 2018 | two critics, taking the smaller of their estimates, against the overestimation of Notebook 10 |
| **SAC** | 2018 | a random policy that is also rewarded for its **entropy**, which gives steady exploration |

All three are **off-policy**, like DQN, and all three reuse what Module C built. **Soft actor-critic (SAC)** is the usual first choice today. We take it from Stable-Baselines3, where it is used exactly like the PPO of Notebook 14.

**This cell takes a few minutes**, although it uses only 6,000 steps of experience: SAC performs a gradient update on a minibatch after every single step.

In [ ]:
from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy

start = time.time()
sac = SAC("MlpPolicy", "Pendulum-v1", seed=0, verbose=0)
sac_steps, sac_scores = [], []
for chunk in range(6):
    sac.learn(total_timesteps=1000, reset_num_timesteps=False)
    score, spread = evaluate_policy(sac, gym.make("Pendulum-v1"), n_eval_episodes=5)
    sac_steps.append(1000 * (chunk + 1))
    sac_scores.append(score)
    print(f"SAC after {sac_steps[-1]:>5,} steps: average total reward {score:7.0f}")
print(f"training took {time.time() - start:.0f} seconds")

In [ ]:
ppo_steps = 200 * np.arange(20, len(returns) + 1)
ppo_curve = np.convolve(returns, np.ones(20) / 20, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(ppo_steps, ppo_curve, label="our PPO (average of 20 training episodes)")
plt.plot(sac_steps, sac_scores, marker="o", label="SAC (evaluation)")
plt.xscale("log")
plt.title("Experience needed on the pendulum")
plt.xlabel("environment steps (logarithmic scale)"); plt.ylabel("total reward of an episode")
plt.grid(True); plt.legend()
plt.show()

good = -300
ppo_reached = ppo_steps[np.argmax(ppo_curve > good)] if (ppo_curve > good).any() else None
sac_reached = next((s for s, score in zip(sac_steps, sac_scores) if score > good), None)
print(f"first reached an average above {good}:  PPO after {ppo_reached} steps,  SAC after {sac_reached} steps")

The horizontal axis is logarithmic, and the two curves are far apart on it: SAC reaches a good swing-up after a few thousand steps, several times less experience than PPO needs. It pays for that in computation, with one gradient step per environment step, so on the clock it was the slower of the two here.

That is the trade-off between the two families, in one picture:

| | On-policy (PPO) | Off-policy (SAC, TD3) |
|---|---|---|
| Experience needed | much | little |
| Computation per step of experience | little | much |
| Stability, ease of tuning | robust | more settings, more that can go wrong |
| Best when | simulation is cheap and can run in parallel | every step of experience is expensive |

> ⚠️ **Common pitfalls**
>
> - **Forgetting the bounds.** A sampled action outside the allowed range must be clipped before it reaches the environment.
> - **Not summing the log-probability over action dimensions.** With more than one dimension the ratio in PPO would have the wrong shape and the wrong meaning.
> - **A spread that starts too small.** The agent then never tries torques far from its first guess. Exercise 2 looks at this.
> - **Treating a time limit as a terminal state.** In the pendulum every episode is truncated. Dropping the value of the last state would tell the critic that the future is always worth zero.
> - **Evaluating with sampled actions.** For a final score, act at the mean. The scatter is for learning.

---
## ✏️ Exercises

The exercises use `GaussianActor`, `ppo_continuous`, `evaluate`, `mean_action`, `record` and the results of Sections 4 and 6.

### Exercise 1 (The log-probability of a torque)
A Gaussian policy has mean 0.5 and standard deviation 0.4 in some state, and the sampled action is 1.0. Compute its log-probability from the formula of the normal density, $\log p(a) = -\frac{(a-\mu)^2}{2\sigma^2} - \log\sigma - \frac12\log(2\pi)$, and check with `torch.distributions.Normal`. Then compute it again for standard deviations of 0.1 and 2. For which spread is this action most probable, and why is it not the smallest?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
mu, action = 0.5, 1.0
for sigma in (0.4, 0.1, 2.0):
    by_hand = -(action - mu) ** 2 / (2 * sigma ** 2) - np.log(sigma) - 0.5 * np.log(2 * np.pi)
    library = torch.distributions.Normal(mu, sigma).log_prob(torch.tensor(action)).item()
    print(f"sigma = {sigma}: by hand {by_hand:7.3f}   PyTorch {library:7.3f}")
```

*With a very small spread the density is concentrated tightly around the mean, and an action half a unit away is extremely unlikely. With a very large spread the density is thin everywhere. The action is most probable for a spread comparable to its distance from the mean. This is why the gradient can both widen and narrow the policy: it widens when good actions keep turning up far from the mean, and narrows when they are close to it.*
</details>

### Exercise 2 (How much scatter at the start?)
Train on the pendulum for 80,000 steps with `initial_std` = 0.1 and 3.0 (seed 0) and compare with the notebook's value of 1.0: the evaluated score of the final policy, and the standard deviation at the end of training. Explain each result.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
print(f"initial std 1.0: final score {evaluate(mean_action(actor)):6.0f},  std at the end {spreads[-1]:.2f}")
for initial_std in (0.1, 3.0):
    other_actor, other_returns, other_spreads = ppo_continuous("Pendulum-v1", 80000, initial_std=initial_std)
    print(f"initial std {initial_std}: final score {evaluate(mean_action(other_actor)):6.0f},  "
          f"std at the end {other_spreads[-1]:.2f}")
```

*A very small initial spread fails: the agent hardly varies its torque, so it learns little about what other torques would do, and since nothing it tries is much better than its first guess, the spread has no reason to grow. A large initial spread did no harm in our run. Many sampled torques are clipped to ±2 at first, but the agent still sees the whole range, and the spread narrows by itself once it knows what works. The spread is the continuous counterpart of epsilon: starting too wide costs a little time, starting too narrow can cost everything.*
</details>

### Exercise 3 (The horizon)
The notebook used a discount factor of 0.9. Train for 80,000 steps with `gamma=0.99` (seed 0) and compare the evaluated score. Then print $1/(1-\gamma)$ for both values: it is a rough measure of how many steps ahead the agent looks. Why might the shorter horizon be easier to learn here?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
far_actor, far_returns, far_spreads = ppo_continuous("Pendulum-v1", 80000, gamma=0.99)
print(f"gamma = 0.9:  final score {evaluate(mean_action(actor)):6.0f}   horizon about {1 / (1 - 0.9):.0f} steps")
print(f"gamma = 0.99: final score {evaluate(mean_action(far_actor)):6.0f}   horizon about {1 / (1 - 0.99):.0f} steps")
```

*In our run the two scores are close, so on this task the choice is not critical. The argument for the shorter horizon is about the critic. With gamma = 0.99 the value of a state is a sum over roughly a hundred future rewards, each between −16 and 0: large numbers that vary a lot and are harder to fit. With 0.9 the values are sums over about ten steps. The pendulum's reward is given at every step, so little is lost by not looking further. On Lunar Lander (Notebook 14), where the decisive reward comes at the very end, a discount factor close to 1 was needed.*
</details>

### Exercise 4 (Another off-policy algorithm)
Train TD3 from Stable-Baselines3 on the pendulum in the same way as SAC in Section 6 (`from stable_baselines3 import TD3`, 6 chunks of 1,000 steps, evaluation after each). Plot its curve together with SAC's. **This takes a few minutes.** Do both reach a good score within 6,000 steps?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
from stable_baselines3 import TD3

td3 = TD3("MlpPolicy", "Pendulum-v1", seed=0, verbose=0)
td3_scores = []
for chunk in range(6):
    td3.learn(total_timesteps=1000, reset_num_timesteps=False)
    td3_scores.append(evaluate_policy(td3, gym.make("Pendulum-v1"), n_eval_episodes=5)[0])
    print(f"TD3 after {1000 * (chunk + 1):>5,} steps: average total reward {td3_scores[-1]:7.0f}")

plt.figure(figsize=(7, 4))
plt.plot(sac_steps, sac_scores, marker="o", label="SAC")
plt.plot(sac_steps, td3_scores, marker="o", label="TD3")
plt.title("Two off-policy algorithms on the pendulum")
plt.xlabel("environment steps"); plt.ylabel("total reward of an episode"); plt.grid(True); plt.legend()
plt.show()
```

*Both learn from a few thousand steps, far fewer than PPO needed. With its default settings TD3 explores by adding fixed noise to a deterministic action, while SAC learns how random to be, and on this task SAC usually gets going sooner. One run of each is not a ranking.*
</details>

### Exercise 5 (A lander with a throttle (a little harder))
Lunar Lander has a continuous version: `gym.make("LunarLander-v3", continuous=True)`. Its action is two numbers between −1 and 1, the throttle of the main engine and of the side engines. Train `ppo_continuous` on it for 600,000 steps with `gamma=0.999, lam=0.98, epochs=4` and `env_kwargs=dict(continuous=True)`. **This takes several minutes.** Print the average return of the last 50 episodes and compare with the 238 of the discrete agent in Notebook 14. Suggest reasons for the difference.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
throttle_actor, throttle_returns, throttle_spreads = ppo_continuous(
    "LunarLander-v3", 600000, gamma=0.999, lam=0.98, epochs=4, learning_rate=3e-4,
    env_kwargs=dict(continuous=True), log=40)
print(f"\naverage return of the last 50 episodes: {throttle_returns[-50:].mean():.0f}")
print(f"standard deviation of the policy at the end: {throttle_spreads[-1]:.2f}")
```

*In our runs the continuous agent learned to come down under control but stayed well below 200 after the same 600,000 steps. Look at the standard deviation at the end: it is still large. A policy that scatters its throttle widely keeps firing engines it does not need, which costs fuel and reward, and precise control only appears as the spread narrows. With four possible buttons the discrete agent had far less to explore. Continuous control gives finer behaviour in the end and usually takes more experience to get there.*
</details>

## 🔑 Key takeaways

- With **continuous actions** there is no list to take a maximum over, so DQN does not apply.
- A **Gaussian policy** outputs the mean of a normal distribution and learns its spread. Acting is sampling, and the log-probability has a closed form.
- **PPO** carries over with two changed lines: the distribution, and a sum of log-probabilities over the action dimensions.
- The policy's **standard deviation** is its exploration, and it shrinks by itself as the agent becomes confident.
- **Off-policy** methods for continuous control (DDPG, TD3, SAC) train a critic $Q(s, a)$ and push the actor along its gradient, with a replay buffer.
- On the pendulum SAC needed **several times less experience** than PPO, at the price of far more computation per step.

### 🏁 You have completed Module D!

You can now train agents in both families. Value-based: Q-learning and DQN with its refinements. Policy-based: REINFORCE, actor-critic and PPO, for discrete and continuous actions. You have seen where each is strong, and you have measured it yourself.

Every agent so far learned alone, by reacting to what its environment did. **Module E** looks at what else there is: agents that **plan** with a model of the game, agents that face an **opponent** who is also learning, and the practice of doing reinforcement learning carefully.

---
*End of Module D.*

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*